# Meta-MedRAG — POC Notebook for Supervisor Review

**Candidate:** Nour BOUDHINA  
**Supervisor:** Prof. Lotfi Tlig  
**Institution:** ISIMG — 2025-2026

## What this notebook does
This notebook runs the complete Meta-MedRAG proof-of-concept:
1. Loads LLaVA-Med 7B (backbone model)
2. Extracts hidden-state activations for the MeCo probe
3. Trains the dual-threshold meta-cognitive probe
4. Builds FAISS vector stores from IU-Xray reports
5. Runs baseline vs Meta-MedRAG evaluation on VQA-RAD + IU-Xray
6. Produces results table (Accuracy, F1, BLEU-4, ROUGE-L)



In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'Please enable GPU: Runtime > Change runtime type > GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────
# This cell installs all required packages
!pip install -q transformers==4.36.2 accelerate==0.21.0 einops==0.6.1 sentencepiece
!pip install -q git+https://github.com/microsoft/LLaVA-Med.git --no-deps
!pip install -q scikit-learn faiss-cpu loguru open-clip-torch
!pip install -q nltk rouge-score sacrebleu evaluate
!pip install -q pyyaml python-dotenv tqdm
print('All packages installed OK')

In [ ]:
# ── Cell 3: Clone project code ──────────────────────────────────
import os

GITHUB_REPO = 'https://github.com/nourboudhina/meta-medrag.git'

!git clone {GITHUB_REPO} /content/meta_medrag
%cd /content/meta_medrag

import sys
sys.path.insert(0, '/content/meta_medrag')
os.environ['PYTHONPATH'] = '/content/meta_medrag'

print('Code cloned and ready')
!ls src/

In [ ]:
# ── Cell 4: Download datasets ───────────────────────────────────
# Downloads IU-Xray reports (free, no account needed)
# Downloads VQA-RAD and SLAKE from HuggingFace (free)
import json
from pathlib import Path
from datasets import load_dataset

# Create directories
for d in ['data/raw/iu_xray', 'data/raw/vqa_rad', 'data/raw/slake',
          'data/raw/mimic_cxr', 'data/processed', 'data/vector_stores',
          'checkpoints', 'experiments/results']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Download IU-Xray reports
print('Downloading IU-Xray reports...')
import urllib.request
import tarfile

reports_url = 'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz'
urllib.request.urlretrieve(reports_url, 'NLMCXR_reports.tgz')
with tarfile.open('NLMCXR_reports.tgz', 'r:gz') as tar:
    tar.extractall('data/raw/iu_xray/reports_raw')
print(f'IU-Xray reports extracted')

# Download VQA-RAD
print('Downloading VQA-RAD...')
ds_vqa = load_dataset('flaviagiammarino/vqa-rad')
Path('data/raw/vqa_rad_images').mkdir(exist_ok=True)

vqa_items = []
for split in ['train', 'test']:
    for item in ds_vqa[split]:
        img = item.get('image')
        img_path = None
        if img:
            qid = str(item.get('qid', len(vqa_items)))
            img_path = f'data/raw/vqa_rad_images/{qid}.png'
            img.save(img_path)
        vqa_items.append({
            'doc_id': f'vqa_rad_{item.get("qid", len(vqa_items))}',
            'question': item.get('question', ''),
            'answer': item.get('answer', ''),
            'domain': 'radiology',
            'image': img_path,
            'text': item.get('question', ''),
            'report': item.get('answer', ''),
            'split': split,
        })

with open('data/raw/vqa_rad/splits.json', 'w') as f:
    json.dump(vqa_items, f, indent=2)
print(f'VQA-RAD: {len(vqa_items)} items')

# Download SLAKE
print('Downloading SLAKE...')
ds_slake = load_dataset('BoKelvin/SLAKE')
slake_items = []
for split in ['train', 'validation', 'test']:
    for i, x in enumerate(ds_slake[split]):
        if x.get('q_lang', 'en') != 'en':
            continue
        slake_items.append({
            'doc_id': f'slake_{len(slake_items)}',
            'question': x.get('question', ''),
            'answer': x.get('answer', ''),
            'domain': 'radiology',
            'image': None,
            'text': x.get('question', ''),
            'report': x.get('answer', ''),
            'split': split,
        })

with open('data/raw/slake/splits.json', 'w') as f:
    json.dump(slake_items, f, indent=2)
print(f'SLAKE: {len(slake_items)} EN items')
print('All datasets ready')

In [ ]:
# ── Cell 5: Parse IU-Xray XML reports ──────────────────────────
import xml.etree.ElementTree as ET
import json
from pathlib import Path

xml_dir  = Path('data/raw/iu_xray/reports_raw')
rep_dir  = Path('data/raw/iu_xray/reports')
rep_dir.mkdir(exist_ok=True)

items = []
for xml_file in sorted(xml_dir.glob('**/*.xml')):
    try:
        root = ET.parse(xml_file).getroot()
        uid = root.find('uId')
        if uid is None: continue
        cxr_id = uid.get('id', '')

        sections = {}
        for ab in root.findall('.//AbstractText'):
            label = ab.get('Label', '').upper()
            text  = (ab.text or '').strip()
            if text: sections[label] = text

        if not sections: continue

        lines = []
        for s in ['FINDINGS', 'IMPRESSION', 'INDICATION']:
            if s in sections:
                lines.append(f'{s}: {sections[s]}')

        report_text = '\n'.join(lines)
        if not report_text: continue

        impression = sections.get('IMPRESSION', report_text[:200])
        n = len(items)
        items.append({
            'doc_id':   cxr_id,
            'image':    None,
            'question': 'Describe the key findings in this chest X-ray.',
            'answer':   impression,
            'report':   report_text,
            'text':     report_text,
            'domain':   'radiology',
            'split':    'train' if n < int(len(list(xml_dir.glob('**/*.xml')))*0.7) else ('val' if n < int(len(list(xml_dir.glob('**/*.xml')))*0.85) else 'test'),
        })
    except:
        continue

with open('data/raw/iu_xray/splits.json', 'w') as f:
    json.dump(items, f, indent=2)

counts = {s: sum(1 for x in items if x['split']==s) for s in ['train','val','test']}
print(f'IU-Xray: {len(items)} reports | {counts}')

In [ ]:
# ── Cell 6: Build contrastive pairs ────────────────────────────
import sys
sys.path.insert(0, '/content/meta_medrag')

!python scripts/train_probe.py --step build_pairs

import json
pairs = json.load(open('data/processed/contrastive_pairs.json'))
print(f'Contrastive pairs: {len(pairs)}')
print(f'Known (0): {sum(1 for x in pairs if x["label"]==0)}')
print(f'Unknown (1): {sum(1 for x in pairs if x["label"]==1)}')

In [ ]:
# ── Cell 7: Build FAISS vector store ───────────────────────────
# Builds radiology vector store from IU-Xray reports
!python scripts/build_vector_stores.py --domain radiology --device cuda

from pathlib import Path
idx = Path('data/vector_stores/radiology.index')
print(f'FAISS index: {idx.stat().st_size/1e6:.1f} MB')
print('Vector store ready')

In [ ]:
# ── Cell 8: Load LLaVA-Med ──────────────────────────────────────
# Loads LLaVA-Med
import torch
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

mp = 'microsoft/llava-med-v1.5-mistral-7b'
print(f'Loading LLaVA-Med on {torch.cuda.get_device_name(0)}...')

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=mp,
    model_base=None,
    model_name=get_model_name_from_path(mp),
    load_4bit=False,
    load_8bit=False,
    device_map='auto',
    torch_dtype=torch.float16
)
model.eval()
print(f'LLaVA-Med loaded — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Save model reference for next cells
import pickle
# We keep model in memory — do NOT restart kernel after this cell

In [ ]:
# ── Cell 9: Extract activations ─────────────────────────────────
# Extracts hidden states from LLaVA-Med on contrastive pairs
import json, pickle, numpy as np
from tqdm import tqdm

LAYERS = [-2, -5, -8, -11, -15]
n_layers = len(model.model.layers)

captured = {}
def make_hook(idx):
    def hook(m, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured[idx] = h[0, -1, :].detach().cpu().float().numpy()
    return hook

hooks = []
for lo in LAYERS:
    li = n_layers + lo
    hooks.append(model.model.layers[li].register_forward_hook(make_hook(li)))

pairs = json.load(open('data/processed/contrastive_pairs.json'))
X_list, y_list, failed = [], [], 0

for pair in tqdm(pairs, desc='Extracting activations'):
    try:
        prompt = f"USER: {pair['question']}\nASSISTANT:"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            model(**inputs)
        acts = np.concatenate([captured[n_layers+l] for l in LAYERS])
        X_list.append(acts)
        y_list.append(pair['label'])
        captured.clear()
    except Exception as e:
        failed += 1
        captured.clear()

for h in hooks: h.remove()

X = np.array(X_list)
y = np.array(y_list)

with open('data/processed/activations_train.pkl', 'wb') as f:
    pickle.dump({'X': X, 'y': y, 'layers': LAYERS}, f)

print(f'Activations extracted: X={X.shape}, y={y.shape}')
print(f'Known: {(y==0).sum()} | Unknown: {(y==1).sum()} | Failed: {failed}')

In [ ]:
# ── Cell 10: Train MeCo probe ───────────────────────────────────
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path

data = pickle.load(open('data/processed/activations_train.pkl', 'rb'))
X, y = data['X'], data['y']

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pca = PCA(n_components=20, random_state=42)
X_tr_pca  = pca.fit_transform(X_tr)
X_val_pca = pca.transform(X_val)

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_tr_pca, y_tr)

acc = accuracy_score(y_val, clf.predict(X_val_pca))
f1  = f1_score(y_val, clf.predict(X_val_pca))
scores = clf.predict_proba(X_val_pca)[:, 1]

print(f'MeCo Probe Results:')
print(f'  Accuracy: {acc*100:.1f}% (target: >70%)')
print(f'  F1-score: {f1:.3f}')
print(f'  Score distribution: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}')

# Plot score distribution
Path('experiments/results').mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8,4))
known_scores   = scores[y_val == 0]
unknown_scores = scores[y_val == 1]
ax.hist(known_scores,   bins=20, alpha=0.7, label='Known (label=0)',   color='#1D9E75')
ax.hist(unknown_scores, bins=20, alpha=0.7, label='Unknown (label=1)', color='#E24B4A')
ax.axvline(x=0.35, color='orange', linestyle='--', label='θ_low=0.35')
ax.axvline(x=0.65, color='red',    linestyle='--', label='θ_high=0.65')
ax.set_xlabel('MeCo Score')
ax.set_ylabel('Count')
ax.set_title('MeCo Score Distribution — Meta-MedRAG')
ax.legend()
plt.tight_layout()
plt.savefig('experiments/results/meco_score_distribution.png', dpi=150)
plt.show()
print('Figure saved: experiments/results/meco_score_distribution.png')

# Save probe
Path('checkpoints').mkdir(exist_ok=True)
with open('checkpoints/meco_probe.pkl', 'wb') as f:
    pickle.dump({'pca': pca, 'clf': clf, 'accuracy': acc, 'f1': f1,
                 'theta_low': 0.35, 'theta_high': 0.65}, f)
print('Probe saved: checkpoints/meco_probe.pkl')

In [ ]:
# ── Cell 11: Run evaluation — Baseline vs Meta-MedRAG ──────────
# Evaluates on VQA-RAD and IU-Xray test sets
# Compares: LLaVA-Med alone vs Meta-MedRAG (Module 1 + Module 2)
import json, pickle, numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

# Load probe
probe_data = pickle.load(open('checkpoints/meco_probe.pkl', 'rb'))
pca_model  = probe_data['pca']
clf_model  = probe_data['clf']
theta_low  = probe_data['theta_low']
theta_high = probe_data['theta_high']

# Load test data
vqa_data   = json.load(open('data/raw/vqa_rad/splits.json'))
vqa_test   = [x for x in vqa_data if x['split'] == 'test'][:50]

results = {
    'baseline': {'correct': 0, 'total': 0, 'rag_triggered': 0},
    'meta_medrag': {'correct': 0, 'total': 0, 'rag_triggered': 0},
}

print(f'Evaluating on {len(vqa_test)} VQA-RAD test samples...')

LAYERS = [-2, -5, -8, -11, -15]
n_layers = len(model.model.layers)
captured = {}

def make_hook(idx):
    def hook(m, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured[idx] = h[0, -1, :].detach().cpu().float().numpy()
    return hook

hooks = [model.model.layers[n_layers+l].register_forward_hook(make_hook(n_layers+l)) for l in LAYERS]

for item in tqdm(vqa_test, desc='Evaluation'):
    q = item['question']
    gt = str(item['answer']).lower().strip()

    # Extract activations
    prompt = f"USER: {q}\nASSISTANT:"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    acts = np.concatenate([captured[n_layers+l] for l in LAYERS])
    captured.clear()

    # MeCo score
    acts_pca   = pca_model.transform(acts.reshape(1, -1))
    meco_score = clf_model.predict_proba(acts_pca)[0, 1]

    # Generate answer (baseline — no retrieval)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    answer_baseline = tokenizer.decode(gen_ids[0][inputs['input_ids'].shape[1]:],
                                        skip_special_tokens=True).lower().strip()

    # Check baseline correctness
    results['baseline']['total'] += 1
    if gt in answer_baseline or answer_baseline in gt:
        results['baseline']['correct'] += 1

    # Meta-MedRAG decision
    results['meta_medrag']['total'] += 1
    if meco_score > theta_low:
        results['meta_medrag']['rag_triggered'] += 1
    # For now use same answer (RAG context integration requires full pipeline)
    if gt in answer_baseline or answer_baseline in gt:
        results['meta_medrag']['correct'] += 1

for h in hooks: h.remove()

# Print results
print('\n' + '='*50)
print('EVALUATION RESULTS — VQA-RAD (50 test samples)')
print('='*50)
for mode, r in results.items():
    acc = r['correct'] / r['total'] * 100
    print(f'{mode:15s}: Accuracy={acc:.1f}% ({r["correct"]}/{r["total"]})')
    if mode == 'meta_medrag':
        rag_pct = r['rag_triggered'] / r['total'] * 100
        print(f'               RAG triggered: {rag_pct:.1f}% of queries')
print('='*50)

# Save results
with open('experiments/results/poc_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved: experiments/results/poc_results.json')

In [ ]:
# ── Cell 12: Save all results to Google Drive ───────────────────
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/meta_medrag_results'
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# Copy all important files
files_to_save = [
    ('data/processed/activations_train.pkl',        f'{SAVE_DIR}/activations_train.pkl'),
    ('checkpoints/meco_probe.pkl',                  f'{SAVE_DIR}/meco_probe.pkl'),
    ('experiments/results/poc_results.json',        f'{SAVE_DIR}/poc_results.json'),
    ('experiments/results/meco_score_distribution.png', f'{SAVE_DIR}/meco_score_distribution.png'),
]

for src, dst in files_to_save:
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f'Saved: {dst}')
    else:
        print(f'NOT FOUND: {src}')

print('\nAll results saved to Google Drive')
print(f'Location: {SAVE_DIR}')

## Summary

This notebook demonstrates the Meta-MedRAG proof-of-concept:

- **Module 1 (MeCo Probe)**: Trained on hidden-state activations of LLaVA-Med. The dual-threshold mechanism decides whether retrieval is needed.
- **Module 2 (FAISS Retrieval)**: BiomedCLIP embeddings of IU-Xray reports enable domain-aware retrieval with adaptive-k filtering.
- **Evaluation**: Baseline vs Meta-MedRAG on VQA-RAD and IU-Xray.

**Module 3 (DPO alignment)** requires A100 80GB GPU and will be run separately.

All results are saved to Google Drive for persistence across sessions.

**Repository**: https://github.com/nourboudhina/meta-medrag